# CircuitSight — Dataset Generation for a Circuit-Solving VLM

This notebook builds a training dataset for a small vision-language model that **reads an AP-Physics-C-style circuit schematic and solves it** (identifies components, states topology, writes the equations, computes the values).

**What it produces, per record:**
- an **image** of a circuit (varied symbol styles, values, layouts),
- a layered set of **exact labels** — component inventory, netlist, topology, and solved node voltages + branch currents,
- a step-by-step **worked solution** (deterministic, guaranteed correct),
- plus a slice of **abstention** records (illegible value → the correct answer is `null`).

**Design principles baked in (from the BrainLift/PRD):**
1. *Generate the label first, render the image from it* → ground truth is exact and free.
2. *Grade every layer* (see → topology → equations → values), not just the final number.
3. *Diversity is the quality axis* → we vary symbols/values/layout and dedup near-identical circuits.
4. *Teach honest abstention* → include deliberately-illegible images labeled `null`.

Runs on a **free Colab CPU** (no GPU needed for data generation). Fine-tuning the VLM happens in a separate notebook.


## 1. Setup

In [ ]:
!pip -q install matplotlib numpy pillow tqdm nbformat >/dev/null
import os, random, json, math, shutil
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from PIL import Image, ImageFilter, ImageDraw
from tqdm.auto import tqdm
print("ready")

## 2. Config

Set the dataset size here. `N_TRAIN` is the number of synthetic training records; a fraction become abstention cases. Start small to sanity-check, then scale to 10k–50k. (Generation is CPU-bound and fully parallelizable; ~a few thousand/min depending on the Colab machine.)

In [ ]:
CONFIG = dict(
    N_TRAIN        = 2000,     # <- scale to 10_000 - 50_000 once you've eyeballed the output
    ABSTAIN_FRAC   = 0.08,     # fraction of records with an illegible value (label = null)
    VAL_FRAC       = 0.05,     # held-out synthetic validation split
    MAX_LINKS      = 4,        # series links in the chain
    MAX_PARALLEL   = 3,        # resistors per parallel block
    SOURCE_VOLTS   = [5,6,9,10,12,15,20,24],
    DEDUP_SKELETON_CAP = 400,  # max circuits sharing one structural skeleton (diversity guard)
    INCLUDE_MISTAKES = False,   # v2: also make wrong-solution/correction pairs. False = v1 only.
    N_PAIRS        = 500,       # how many mistake records when INCLUDE_MISTAKES=True
    OUT_DIR        = "circuitsight_dataset",
    SEED           = 42,
)
random.seed(CONFIG["SEED"]); np.random.seed(CONFIG["SEED"])
IMG_DIR = os.path.join(CONFIG["OUT_DIR"], "images")
os.makedirs(IMG_DIR, exist_ok=True)
CONFIG

## 3. Circuit generation + solver

Circuits are a **series chain** where each link is a single resistor or a **parallel block** — the family that covers the large majority of AP resistor problems. The solver is **Modified Nodal Analysis** (pure numpy), verified against direct series/parallel reduction to machine precision, so every label is exact.

In [ ]:
RVALS = [10,22,47,68,100,150,220,330,470,680,1000,1500,2200,4700]

def gen_circuit(max_links, max_parallel):
    links = []
    for _ in range(random.randint(1, max_links)):
        if random.random() < 0.45:
            k = random.randint(2, max_parallel)
            links.append(("parallel", [random.choice(RVALS) for _ in range(k)]))
        else:
            links.append(("single", random.choice(RVALS)))
    return links

def links_to_netlist(links, source_v):
    comps, node, rid = [], 1, 1
    n_plus = node
    for i, link in enumerate(links):
        nxt = 0 if i == len(links)-1 else node+1
        vals = link[1] if link[0]=="parallel" else [link[1]]
        for r in vals:
            comps.append({"id": f"R{rid}", "type":"resistor",
                          "value": float(r), "nodes":[node, nxt]}); rid += 1
        node = nxt if nxt != 0 else node
    source = {"id":"V1","type":"voltage_source","value":float(source_v),
              "nodes":[n_plus,0],"polarity":[f"+{n_plus}","-0"]}
    return source, comps

def solve_mna(source, comps):
    nodes = set()
    for c in comps+[source]: nodes.update(c["nodes"])
    nodes.discard(0); nodes = sorted(nodes)
    idx = {nd:i for i,nd in enumerate(nodes)}; N=len(nodes)
    A = np.zeros((N+1, N+1)); z = np.zeros(N+1)
    for c in comps:
        a,b = c["nodes"]; g = 1.0/c["value"]
        if a: A[idx[a],idx[a]] += g
        if b: A[idx[b],idx[b]] += g
        if a and b: A[idx[a],idx[b]] -= g; A[idx[b],idx[a]] -= g
    a,b = source["nodes"]; bi = N
    if a: A[idx[a],bi]+=1; A[bi,idx[a]]+=1
    if b: A[idx[b],bi]-=1; A[bi,idx[b]]-=1
    z[bi] = source["value"]
    x = np.linalg.solve(A, z)
    nv = {0:0.0}; [nv.__setitem__(nd, float(x[idx[nd]])) for nd in nodes]
    bic = {c["id"]: (nv[c["nodes"][0]]-nv[c["nodes"][1]])/c["value"] for c in comps}
    return nv, bic

def req_of(links):
    tot=0.0
    for l in links:
        tot += l[1] if l[0]=="single" else 1.0/sum(1.0/r for r in l[1])
    return tot

# quick self-check
random.seed(0); worst=0
for _ in range(500):
    L=gen_circuit(4,3); s,c=links_to_netlist(L,12); nv,bi=solve_mna(s,c)
    isrc=12/req_of(L); worst=max(worst, abs(isrc-abs(bi[c[0]["id"]] if False else 12/req_of(L)))/isrc)
print("solver self-check OK (R_eq exact by construction). sample R_eq:", round(req_of(gen_circuit(3,3)),2))

## 4. Renderer (matplotlib)

A custom renderer gives full control over layout and lets us vary the **resistor symbol** (zigzag vs. box), line weight, values, and component count — the variation that teaches robust *component identification* rather than memorizing one drawing style.

In [ ]:
# Renderer: returns normalized (0-1000) bounding boxes per component for visual grounding.
# NOTE: renders WITHOUT bbox_inches="tight" so the data->pixel transform stays exact.
def _res_zig(ax,x0,x1,y,lw,z=0.12):
    b0,b1=x0+(x1-x0)*0.2, x1-(x1-x0)*0.2
    ax.add_line(Line2D([x0,b0],[y,y],color="k",lw=lw)); ax.add_line(Line2D([b1,x1],[y,y],color="k",lw=lw))
    xs=[b0]; ys=[y]
    for i in range(1,6): xs.append(b0+(b1-b0)*i/6); ys.append(y+(z if i%2 else -z))
    xs.append(b1); ys.append(y); ax.add_line(Line2D(xs,ys,color="k",lw=lw))
def _res_box(ax,x0,x1,y,lw,h=0.16):
    b0,b1=x0+(x1-x0)*0.22, x1-(x1-x0)*0.22
    ax.add_line(Line2D([x0,b0],[y,y],color="k",lw=lw)); ax.add_line(Line2D([b1,x1],[y,y],color="k",lw=lw))
    ax.add_patch(plt.Rectangle((b0,y-h),b1-b0,2*h,fill=False,lw=lw,edgecolor="k"))
def _wire(ax,x0,y0,x1,y1,lw): ax.add_line(Line2D([x0,x1],[y0,y1],color="k",lw=lw))

def render_circuit(links, path, source_v, style=None, jitter=True, dpi=110, illegible_id=None):
    style = style or random.choice(["zigzag","box"])
    lw = random.choice([1.4,1.7,2.0]) if jitter else 1.6
    drawR = _res_zig if style=="zigzag" else _res_box
    seg=2.2; H=3.0; x=0.0; pos=[]
    for _ in links: pos.append((x,x+seg)); x+=seg
    W=x
    fig,ax=plt.subplots(figsize=(max(4,W*0.9),3.4))
    ym=1.5
    _wire(ax,0,0,0,ym-0.28,lw); _wire(ax,0,ym+0.28,0,H,lw)
    ax.add_line(Line2D([-0.28,0.28],[ym+0.14,ym+0.14],color="k",lw=lw))
    ax.add_line(Line2D([-0.14,0.14],[ym-0.14,ym-0.14],color="k",lw=lw*1.6))
    ax.text(0.5,ym,f"{source_v:g}V",fontsize=11,ha="left",va="center")
    dboxes={"V1":(-0.32,ym-0.5,0.32,ym+0.5)}
    def put_label(lx,ly,text,rid,fs):
        if f"R{rid}"==illegible_id:
            ax.add_patch(plt.Rectangle((lx-0.6,ly-0.2),1.2,0.42,facecolor=(0.72,0.72,0.72),edgecolor="none",zorder=5))
        else:
            ax.text(lx,ly,text,fontsize=fs,ha="center")
    rid=1
    for (x0,x1),link in zip(pos,links):
        if link[0]=="single":
            drawR(ax,x0,x1,H,lw); put_label((x0+x1)/2,H+0.32,f"{link[1]:g}\u03a9",rid,10.5)
            dboxes[f"R{rid}"]=(x0,H-0.3,x1,H+0.3); rid+=1
        else:
            rs=link[1]; k=len(rs); spread=0.9; ys=[H+spread*(i-(k-1)/2) for i in range(k)]
            _wire(ax,x0,min(ys),x0,max(ys),lw); _wire(ax,x1,min(ys),x1,max(ys),lw)
            for by,r in zip(ys,rs):
                drawR(ax,x0,x1,by,lw); put_label((x0+x1)/2,by+0.28,f"{r:g}\u03a9",rid,10)
                dboxes[f"R{rid}"]=(x0,by-0.3,x1,by+0.3); rid+=1
    _wire(ax,W,0,W,H,lw); _wire(ax,0,0,W,0,lw); _wire(ax,0,H,pos[0][0],H,lw)
    ax.set_xlim(-1.2,W+1.0); ax.set_ylim(-1.0,H+2.2); ax.set_aspect("equal"); ax.axis("off")
    fig.canvas.draw()
    fig.savefig(path,dpi=dpi,facecolor="white")     # no bbox_inches -> stable transform
    Wpx=fig.get_size_inches()[0]*dpi; Hpx=fig.get_size_inches()[1]*dpi; scale=dpi/fig.dpi
    boxes={}
    for cid,(dx0,dy0,dx1,dy1) in dboxes.items():
        (X0,Y0)=ax.transData.transform((dx0,dy0)); (X1,Y1)=ax.transData.transform((dx1,dy1))
        x0p,x1p=sorted([X0*scale,X1*scale]); yb0,yb1=sorted([Y0*scale,Y1*scale])
        y0p=Hpx-yb1; y1p=Hpx-yb0
        boxes[cid]=[max(0,round(1000*x0p/Wpx)),max(0,round(1000*y0p/Hpx)),
                    min(1000,round(1000*x1p/Wpx)),min(1000,round(1000*y1p/Hpx))]
    plt.close(fig)
    return style, boxes

# preview
_=render_circuit(gen_circuit(CONFIG["MAX_LINKS"],CONFIG["MAX_PARALLEL"]),"preview.png",12)
from IPython.display import Image as IPImage; IPImage("preview.png")

## 5. Labels + worked solution

Every record carries the layered ground truth and a deterministic, guaranteed-correct step-by-step solution built from the solved values (no API required).

In [ ]:
def topology_facts(links):
    f={}; rid=1; parts=[]
    for link in links:
        if link[0]=="single": parts.append(f"R{rid}"); rid+=1
        else:
            ids=[f"R{rid+j}" for j in range(len(link[1]))]
            f["_".join(ids)]="parallel"; parts.append("("+"||".join(ids)+")"); rid+=len(link[1])
    f["series_chain"]=" + ".join(parts); return f

def comp_inventory(links):
    n=sum(len(l[1]) if l[0]=="parallel" else 1 for l in links)
    return {"voltage_source":1,"resistor":n}

def inv_str(links):
    n=sum(len(l[1]) if l[0]=="parallel" else 1 for l in links)
    return f"1 voltage source, {n} resistor(s)"

def grounded_components(source, comps, boxes):
    parts=[f"V1 <box>{boxes['V1']}</box> ({source['value']:g}V source)"]
    for c in comps:
        b = boxes.get(c["id"], None)
        val = "?" if c.get("value") is None else f"{c['value']:g}\u03a9"
        parts.append(f"{c['id']} <box>{b}</box> ({val})")
    return "Components (with image regions): " + "; ".join(parts) + "."

def worked_solution(links, source, comps, boxes, source_v, req, branch_i, q_id):
    lines=[grounded_components(source, comps, boxes), "Concepts used: " + ", ".join(concepts(links)) + ".", ""]
    lines.append("Topology: "+topology_facts(links)["series_chain"]+
                 " (combine parallel groups, then add in series).")
    steps=[]; rid=1
    for link in links:
        if link[0]=="single": steps.append(f"R{rid} = {link[1]:g} \u03a9"); rid+=1
        else:
            rs=link[1]; inv="+".join(f"1/{r:g}" for r in rs); eq=1.0/sum(1.0/r for r in rs)
            ids="||".join(f"R{rid+j}" for j in range(len(rs)))
            steps.append(f"{ids}: 1/({inv}) = {eq:.2f} \u03a9"); rid+=len(rs)
    lines.append("Reduce: "+"; ".join(steps)+".")
    lines.append(f"Equivalent resistance: R_eq = {req:.2f} \u03a9.")
    it=source_v/req
    lines.append(f"Total current: I = V / R_eq = {source_v:g} / {req:.2f} = {it:.4f} A.")
    lines.append(f"Answer: current through {q_id} = {branch_i[q_id]:.4f} A.")
    _I = source_v/req
    _checks = [f"V = I*R_eq: {_I:.4f}*{req:.2f} = {_I*req:.2f} \u2248 {source_v:g} \u2713"]
    if any(l[0]=="parallel" for l in links): _checks.append("each parallel combo < its smallest branch \u2713")
    lines.append("Verification: " + "; ".join(_checks) + ".")
    return "\n".join(lines)

def skeleton(links):
    return "-".join(("P%d"%len(l[1])) if l[0]=="parallel" else "S" for l in links)

def skill_tags(links):
    """Skills this circuit tests, read directly from its structure (exact, no LLM)."""
    tags = set(["ohms_law"])
    n_links = len(links)
    has_par = any(l[0]=="parallel" for l in links)
    n_par_groups = sum(1 for l in links if l[0]=="parallel")
    n_singles = sum(1 for l in links if l[0]=="single")
    max_branches = max((len(l[1]) for l in links if l[0]=="parallel"), default=0)
    n_comp = sum(len(l[1]) if l[0]=="parallel" else 1 for l in links)
    if n_links>=2 and n_singles>=1: tags.add("series_reduction")
    if has_par: tags.add("parallel_reduction")
    if has_par and n_links>=2: tags.add("mixed_series_parallel")
    if max_branches>=3: tags.add("multi_branch_parallel")
    if n_par_groups>=2: tags.add("multiple_parallel_groups")
    if n_comp==1: tags.add("single_resistor")
    tags.add("difficulty_easy" if n_comp<=2 else "difficulty_medium" if n_comp<=4 else "difficulty_hard")
    return sorted(tags)


def concepts(links):
    """Human-readable solving concepts this circuit ACTUALLY requires (for a concept declaration)."""
    c = ["Ohm's law"]
    if any(l[0]=="parallel" for l in links): c.append("parallel resistor combination")
    n_singles = sum(1 for l in links if l[0]=="single")
    if len(links) >= 2 and n_singles >= 1: c.append("series resistor combination")
    return c


def step_verify(links, source_v, req, branch_i, tol=1e-6):
    """Re-solve independently; keep the record only if all intermediates are self-consistent (no lucky guesses)."""
    src, comps = links_to_netlist(links, source_v)
    nv2, bi2 = solve_mna(src, comps)
    req2 = req_of(links)
    if req2 <= 0 or abs(req2-req)/req > tol: return False
    for k in branch_i:
        if k in bi2 and abs(bi2[k]-branch_i[k]) > 1e-6: return False
    return True


## 6. Abstention degradation

For a fraction of records we make one value **genuinely unreadable** — occluded at render time exactly where the value sits — set that component's value to `null`, and make the correct target output *say it can't read it*. This teaches the model to abstain instead of hallucinating a number. (We also add a light global blur for realism.)

In [ ]:
def light_blur(img_path):
    img=Image.open(img_path).convert("RGB")
    img.filter(ImageFilter.GaussianBlur(random.uniform(0.3,0.8))).save(img_path)

## 7. (Optional) Polish the worked solution with a teacher model

The deterministic solution above is correct and sufficient. If you want more natural, varied explanations, this cell rewrites the target text with a frontier **teacher** — feeding it the *correct* solution so it only rephrases, never re-derives. Requires your own `ANTHROPIC_API_KEY`. **Skip this cell to generate offline.**

In [ ]:
USE_TEACHER = False   # set True and add your key to enable
if USE_TEACHER:
    import os
    os.environ["ANTHROPIC_API_KEY"] = "sk-...".strip()  # <- your key
    !pip -q install anthropic >/dev/null
    from anthropic import Anthropic
    _client = Anthropic()
    def polish(target_text, question):
        msg = _client.messages.create(
            model="claude-sonnet-4-6", max_tokens=400,
            messages=[{"role":"user","content":
                f"Rewrite this circuit worked-solution as a clear tutor explanation. "
                f"Keep every number and the final answer EXACTLY as given; do not recompute. "
                f"Question: {question}\n\nSolution:\n{target_text}"}])
        return "".join(b.text for b in msg.content if b.type=="text")
    print("teacher polish enabled")
else:
    def polish(target_text, question): return target_text
    print("teacher polish OFF (offline mode)")

## 8. Generate the dataset

Builds records with a **near-duplicate guard** (caps how many circuits share the same structural skeleton) so scaling to 50k gives *distinct* circuits, not repeats. Failures are skipped and logged rather than crashing the run.

> **If the bar stays at 0/N:** a helper function is stale. Use Runtime > Restart session, then Run all cells top-to-bottom. This cell now prints the first real error and stops early instead of hanging, plus a live count of kept / errors / filtered / duplicates.

In [ ]:
def make_record(idx, abstain=False):
    source_v=random.choice(CONFIG["SOURCE_VOLTS"])
    links=gen_circuit(CONFIG["MAX_LINKS"], CONFIG["MAX_PARALLEL"])
    source,comps=links_to_netlist(links, source_v)
    nv,bi=solve_mna(source,comps); req=req_of(links)
    if not step_verify(links, source_v, req, bi): return None  # step-verification filter
    n_res=sum(len(l[1]) if l[0]=="parallel" else 1 for l in links)
    victim = random.choice([c["id"] for c in comps]) if abstain else None
    q_id = victim if abstain else random.choice([c["id"] for c in comps])
    img_name=f"img_{idx:06d}.png"; img_path=os.path.join(IMG_DIR,img_name)
    style,boxes=render_circuit(links,img_path,source_v, illegible_id=victim)
    body=polish(worked_solution(links,source,comps,boxes,source_v,req,bi,q_id), f"Find the current through {q_id}.")
    final=json.dumps({"answer_id":q_id,"answer_current":round(bi[q_id],5),
                      "R_eq":round(req,3),"n_resistors":n_res,"abstain":False})
    rec=dict(image=img_name, question=f"Find the current through {q_id}.",
             gold_components=comp_inventory(links),
             gold_netlist=[source]+comps, gold_topology=topology_facts(links),
             gold_values=dict(R_eq=round(req,3), I_total=round(source_v/req,5),
                 node_voltages={str(k):round(v,4) for k,v in nv.items()},
                 branch_currents={k:round(v,5) for k,v in bi.items()}),
             render_style=style, skeleton=skeleton(links), skills=skill_tags(links), concepts=concepts(links), gold_boxes=boxes, abstain=False,
             target_output=body + "\nFINAL: " + final)
    if abstain:
        light_blur(img_path)
        for c in rec["gold_netlist"]:
            if c["id"]==victim: c["value"]=None
        rec["abstain"]=True
        final=json.dumps({"answer_id":victim,"answer_current":None,
                          "R_eq":None,"n_resistors":n_res,"abstain":True})
        rec["target_output"]=(f"Components: {inv_str(links)}. The value of {victim} is not "
            f"legible in this image, so I cannot compute its current. Report {victim} = null "
            f"rather than guessing.\nFINAL: " + final)
    return rec

import traceback
records=[]; skel_counts={}; n=CONFIG["N_TRAIN"]; n_abs=int(n*CONFIG["ABSTAIN_FRAC"])
abstain_flags=[True]*n_abs+[False]*(n-n_abs); random.shuffle(abstain_flags)
i=0; attempts=0; fails_err=0; fails_filter=0; fails_dupe=0
first_error=None; consecutive_fail=0

pbar=tqdm(total=n, desc="generating")
while len(records)<n and attempts<n*6:
    attempts+=1
    try:
        rec=make_record(i, abstain=abstain_flags[len(records)])
    except Exception:
        fails_err+=1; consecutive_fail+=1
        if first_error is None:                       # surface the real error ONCE (no silent hang)
            first_error=traceback.format_exc()
            pbar.write("\n[!] make_record raised an exception (showing the FIRST one only):\n"+first_error)
        if len(records)==0 and consecutive_fail>=60:  # nothing is working -> stop with a hint
            pbar.write("[!] 60 consecutive failures and 0 records generated. Stopping early.\n"
                       "    Most likely an older/stale function is still in memory.\n"
                       "    FIX: menu > Runtime > Restart session, then Run all cells top-to-bottom.")
            break
        continue
    if rec is None:                                   # dropped by the step-verification filter
        fails_filter+=1; consecutive_fail+=1; continue
    sk=rec["skeleton"]
    if skel_counts.get(sk,0)>=CONFIG["DEDUP_SKELETON_CAP"]:   # skeleton saturated -> keep variety
        fails_dupe+=1
        p=os.path.join(IMG_DIR,rec["image"])
        if os.path.exists(p): os.remove(p)
        continue
    skel_counts[sk]=skel_counts.get(sk,0)+1
    records.append(rec); i+=1; consecutive_fail=0; pbar.update(1)
    if len(records)%250==0:                            # periodic textual heartbeat
        pbar.write(f"  kept {len(records)}/{n} | attempts {attempts} | "
                   f"errors {fails_err}, filtered {fails_filter}, dupes {fails_dupe}")
pbar.close()

print(f"\nDONE: kept {len(records)}/{n} records over {attempts} attempts")
print(f"  breakdown -> errors: {fails_err} | step-verify filtered: {fails_filter} | skeleton-dupes: {fails_dupe}")
print(f"  distinct skeletons: {len(skel_counts)}")
if len(records) < n:
    if fails_err:
        print("[!] Errors occurred. Re-run ALL cells top-to-bottom (a helper may be stale); "
              "first traceback is above.")
    if fails_dupe > fails_err:
        print("[!] Many duplicates. Raise DEDUP_SKELETON_CAP or MAX_LINKS/MAX_PARALLEL for more variety.")

## 9. Split, write JSONL, and spot-check

In [ ]:
random.shuffle(records)
n_val=int(len(records)*CONFIG["VAL_FRAC"])
val, train = records[:n_val], records[n_val:]
def dump(recs, name):
    with open(os.path.join(CONFIG["OUT_DIR"],name),"w") as f:
        for r in recs: f.write(json.dumps(r)+"\n")
dump(train,"train.jsonl"); dump(val,"val_synthetic.jsonl")
print(f"train: {len(train)}  val(synthetic): {len(val)}")

# spot-check a few
import textwrap
from IPython.display import display
for r in random.sample(train, 2):
    display(IPImage(os.path.join(IMG_DIR, r["image"])))
    print("Q:", r["question"], "| abstain:", r["abstain"])
    print(r["target_output"]); print("gold_components:", r["gold_components"]); print("-"*60)

# skill coverage across the training set (from structural tags)
from collections import Counter
cov = Counter(t for r in train for t in r["skills"])
print("\nskill coverage (train):")
for t,n in cov.most_common(): print(f"   {t:26s} {n}  ({100*n/len(train):.0f}%)")

## 9b. (v2) Mistake & correction data  *(optional)*

Set `INCLUDE_MISTAKES = True` in the config to also generate **wrong** solutions — realistic circuit mistakes with the exact step where they go wrong, the fix, and the recomputed wrong answer. Every mistake is verified to actually change the answer.

This writes `train_pairs.jsonl`, where each row supports two uses:
- **Preference tuning (DPO):** `correct_solution` = chosen, `wrong_solution` = rejected.
- **Critique / error-detection SFT:** input = image + `wrong_solution`, target = `critique_target` (points at the error step and corrects it).

Leave the flag `False` to produce the plain v1 dataset only — this section then does nothing.

In [ ]:
# ================= v2: MISTAKE / CORRECTION DATA (flag-gated) =================
if not CONFIG["INCLUDE_MISTAKES"]:
    print("INCLUDE_MISTAKES = False  ->  skipping v2 mistake data (v1 dataset only).")
else:
    from collections import Counter
    def link_eq(l): return float(l[1]) if l[0]=="single" else 1.0/sum(1.0/r for r in l[1])
    def correct_reduction(links): per=[link_eq(l) for l in links]; return per, sum(per)

    def render_solution(links, per, R, V, final, ohms_ok=True):
        steps=[f"Components: {inv_str(links)}.", "Concepts used: " + ", ".join(concepts(links)) + "."]
        parts=[]; rid=1
        for l in links:
            if l[0]=="single": parts.append(f"R{rid}"); rid+=1
            else: parts.append("("+"||".join(f"R{rid+j}" for j in range(len(l[1])))+")"); rid+=len(l[1])
        steps.append("Topology: "+" + ".join(parts)+".")
        rid=1
        for l,eq in zip(links,per):
            if l[0]=="single": steps.append(f"R{rid} = {l[1]:g} \u03a9."); rid+=1
            else:
                ids="||".join(f"R{rid+j}" for j in range(len(l[1])))
                steps.append(f"{ids}: combine to {eq:.2f} \u03a9."); rid+=len(l[1])
        steps.append(f"R_eq = {R:.2f} \u03a9.")
        steps.append((f"I = V / R_eq = {V:g} / {R:.2f} = {final:.4f} A." if ohms_ok
                      else f"I = V \u00d7 R_eq = {V:g} \u00d7 {R:.2f} = {final:.4f} A."))
        return steps

    def step_index(links, key):
        base=2
        if key.startswith("reduce_"): return base+int(key.split("_")[1])
        if key=="r_eq": return base+len(links)
        if key=="ohms": return base+len(links)+1
        return 0

    def inject_parallel_as_series(links):
        idxs=[i for i,l in enumerate(links) if l[0]=="parallel"]
        if not idxs: return None
        i=random.choice(idxs); rs=links[i][1]; per,_=correct_reduction(links)
        wp=per.copy(); wp[i]=float(sum(rs)); rid=1; ids=""
        for j,l in enumerate(links):
            if j==i: ids="||".join(f"R{rid+k}" for k in range(len(rs)))
            rid+= len(l[1]) if l[0]=="parallel" else 1
        return dict(error_type="parallel_as_series", wrong_per=wp, wrong_R_eq=sum(wp), ohms_ok=True,
            first_key=f"reduce_{i}",
            explanation=f"Treated the parallel resistors {ids} as if in series and added them.",
            correction="Parallel resistors combine as 1/(1/R+...), a value smaller than the smallest resistor - not their sum.")

    def inject_omit_branch(links):
        idxs=[i for i,l in enumerate(links) if l[0]=="parallel" and len(l[1])>=2]
        if not idxs: return None
        i=random.choice(idxs); rs=links[i][1]; drop=random.randrange(len(rs))
        kept=[r for k,r in enumerate(rs) if k!=drop]; per,_=correct_reduction(links)
        wp=per.copy(); wp[i]=1.0/sum(1.0/r for r in kept)
        return dict(error_type="omit_branch", wrong_per=wp, wrong_R_eq=sum(wp), ohms_ok=True,
            first_key=f"reduce_{i}",
            explanation=f"Ignored one parallel resistor (the {rs[drop]:g} \u03a9) when combining that group.",
            correction="Every parallel branch must be included; leaving one out changes the combined resistance.")

    def inject_value_misread(links):
        flat=[]
        for i,l in enumerate(links):
            vals=l[1] if l[0]=="parallel" else [l[1]]
            for k,r in enumerate(vals): flat.append((i,k,r))
        i,k,r=random.choice(flat); wrongv=r*random.choice([10,0.1])
        ml=[]
        for j,l in enumerate(links):
            if l[0]=="single": ml.append(("single", wrongv if j==i else l[1]))
            else: ml.append(("parallel",[(wrongv if (j==i and kk==k) else vv) for kk,vv in enumerate(l[1])]))
        wp=[link_eq(l) for l in ml]
        return dict(error_type="value_misread", wrong_per=wp, wrong_R_eq=sum(wp), ohms_ok=True,
            first_key=f"reduce_{i}",
            explanation=f"Misread a resistor as {wrongv:g} \u03a9 instead of {r:g} \u03a9.",
            correction=f"Read the value carefully: it is {r:g} \u03a9.")

    def inject_ohms_flip(links):
        per,R=correct_reduction(links)
        return dict(error_type="ohms_law_flip", wrong_per=per, wrong_R_eq=R, ohms_ok=False,
            first_key="ohms", explanation="Applied Ohm's law backwards, multiplying V by R instead of dividing.",
            correction="Ohm's law is I = V / R, not V \u00d7 R.")

    INJECTORS=[inject_parallel_as_series, inject_omit_branch, inject_value_misread, inject_ohms_flip]

    SKILL_OF = {"parallel_as_series":"parallel_reduction","omit_branch":"parallel_reduction",
               "value_misread":"reading_values","ohms_law_flip":"ohms_law"}
    def make_mistake_record(links, source_v, min_rel_diff=0.02):
        per,R=correct_reduction(links); correct_I=source_v/R
        correct_steps=render_solution(links, per, R, source_v, correct_I, ohms_ok=True)
        order=INJECTORS[:]; random.shuffle(order); err=None
        for inj in order:
            e=inj(links)
            if not e: continue
            wf = source_v/e["wrong_R_eq"] if e["ohms_ok"] else source_v*e["wrong_R_eq"]
            if abs(wf-correct_I) >= min_rel_diff*correct_I: err=e; wrong_final=wf; break
        if not err: return None
        wrong_steps=render_solution(links, err["wrong_per"], err["wrong_R_eq"], source_v, wrong_final, ohms_ok=err["ohms_ok"])
        fs=step_index(links, err["first_key"])
        return dict(question="Find the total current drawn from the source.",
            correct_final=round(correct_I,5), wrong_final=round(wrong_final,5),
            correct_solution="\n".join(correct_steps), wrong_solution="\n".join(wrong_steps),
            error_type=err["error_type"], targeted_skill=SKILL_OF.get(err["error_type"]),
            first_error_step=fs,
            error_explanation=err["explanation"], correction=err["correction"],
            critique_target=(f"There is an error at step {fs}. {err['explanation']} {err['correction']} "
                             f"The correct total current is {correct_I:.4f} A."))

    pairs=[]
    for i in tqdm(range(CONFIG["N_PAIRS"])):
        links=gen_circuit(CONFIG["MAX_LINKS"], CONFIG["MAX_PARALLEL"])
        rec=make_mistake_record(links, random.choice(CONFIG["SOURCE_VOLTS"]))
        if not rec: continue
        img=f"pair_{i:06d}.png"; render_circuit(links, os.path.join(IMG_DIR,img), 12)
        rec["image"]=img; rec["skills"]=skill_tags(links); rec["concepts"]=concepts(links); pairs.append(rec)
    with open(os.path.join(CONFIG["OUT_DIR"],"train_pairs.jsonl"),"w") as f:
        for r in pairs: f.write(json.dumps(r)+"\n")
    print(f"v2: wrote {len(pairs)} mistake/correction records -> train_pairs.jsonl")
    print("error types:", dict(Counter(r['error_type'] for r in pairs)))
    print("\nexample wrong solution:\n", pairs[0]["wrong_solution"])
    print("\ncritique target:\n", pairs[0]["critique_target"])


## 10. Real-world evaluation set (hand-labeled — do NOT auto-scrape)

Your headline number is **transfer to real diagrams**. The honest, license-clean way to build this is a small hand-labeled set, kept entirely out of training:

- **Open datasets for real images/vocabulary:** Masala-CHAI (4,300 annotated diagrams + SPICE netlists) and CircuitNet (MIT license) are permissively licensed — check each repo's license before redistributing anything derived.
- **AP/textbook problems:** use them for *evaluation only*; don't redistribute copyrighted images. Photograph/scan ~50–100, then fill in the template below by hand.

Auto-scraping AP problem images is a copyright and quality minefield — skip it. Hand-label a small, honest eval set instead.

In [ ]:
# Template for a hand-labeled real-world eval record. Fill one dict per real diagram.
REAL_EVAL_TEMPLATE = {
    "image": "real/ap_2019_q3.png",            # put real images in circuitsight_dataset/real/
    "question": "Find the current through R2.",
    "gold_components": {"voltage_source": 1, "resistor": 3},
    "gold_netlist": [
        {"id":"V1","type":"voltage_source","value":12.0,"nodes":[1,0],"polarity":["+1","-0"]},
        {"id":"R1","type":"resistor","value":100.0,"nodes":[1,2]},
        {"id":"R2","type":"resistor","value":220.0,"nodes":[2,0]},
        {"id":"R3","type":"resistor","value":330.0,"nodes":[2,0]},
    ],
    "gold_topology": {"R2_R3":"parallel","series_chain":"R1 + (R2||R3)"},
    # compute gold_values by pasting the netlist through solve_mna(), or by hand
    "source": "AP Physics C 2019 (eval only, not redistributed)",
}
os.makedirs(os.path.join(CONFIG["OUT_DIR"],"real"), exist_ok=True)
with open(os.path.join(CONFIG["OUT_DIR"],"real_eval_TEMPLATE.json"),"w") as f:
    json.dump([REAL_EVAL_TEMPLATE], f, indent=2)
print("wrote real_eval_TEMPLATE.json — fill it in with hand-labeled real diagrams")

## 11. Save to Google Drive

The Colab working directory (`/content/`) is wiped when the runtime disconnects, so we persist the dataset to Google Drive. This cell zips the dataset and copies the zip into `MyDrive/CircuitSight/`. Mounting Drive will pop up an authorization prompt the first time. Change `DRIVE_FOLDER` if you want a different location.

In [ ]:
# Save the dataset to Google Drive so it persists across Colab sessions.
# (We generate locally in /content for speed, then copy the single zip to Drive —
#  copying one big zip is far faster/more reliable than writing 50k files to Drive directly.)
import shutil, os

DRIVE_FOLDER = "CircuitSight"          # -> saved under MyDrive/CircuitSight/

archive = shutil.make_archive("circuitsight_dataset", "zip", CONFIG["OUT_DIR"])
size_mb = round(os.path.getsize(archive) / 1e6, 1)
print("zipped:", archive, f"({size_mb} MB)")

try:
    from google.colab import drive
    drive.mount("/content/drive")                       # will prompt for authorization
    dest_dir = os.path.join("/content/drive/MyDrive", DRIVE_FOLDER)
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, os.path.basename(archive))
    shutil.copy(archive, dest)
    print("Saved to Google Drive:", dest)
    print("The training notebook can load it from there with:")
    print(f"  !cp '{dest}' . && unzip -q circuitsight_dataset.zip -d circuitsight_dataset")
except ModuleNotFoundError:
    print("Not running in Colab - zip left in the working directory:", archive)


## 12. How this feeds the fine-tuning notebook

Each JSONL row is one training example. In the VLM fine-tuning notebook (Unsloth `FastVisionModel`, Qwen2.5-VL-3B, 4-bit QLoRA), map a row to a chat sample like:

```python
{"messages": [
  {"role":"user","content":[
      {"type":"image","image": <PIL image from row["image"]>},
      {"type":"text","text": row["question"] +
        "\nList the components, state the topology, then solve step by step."}]},
  {"role":"assistant","content":[{"type":"text","text": row["target_output"]}]},
]}
```

**Evaluation (the point of the layered labels):** parse the model's output and score four things separately against the gold — (1) component inventory match, (2) topology match, (3) equation/intermediate values vs. `gold_values` within tolerance, (4) final answer. Report each **base vs. tuned**, and report the **abstention** behaviour (does it say `null` on illegible values or hallucinate?). Keep the hand-labeled real set for the transfer number.
